# Custom Multi-Asset Index Construction & Rebalancing with `yfinance`

This notebook builds a **custom momentum-based multi-asset index** using ETF proxies and compares it against the **S&P 500 ETF (`SPY`)**.

## What this notebook includes
- Data download with `yfinance`
- A clear **index methodology**
- **Monthly rebalancing**
- **Momentum-based asset selection**
- Portfolio/index return construction
- Performance comparison vs `SPY`
- Sharpe ratio and other useful metrics
- Clean visualizations

## Strategy idea
We will:
1. Build a **multi-asset universe** using liquid ETFs.
2. Use **trailing 6-month momentum** as the selection signal.
3. Rebalance **monthly**.
4. Select the **top 3 assets** by momentum.
5. Weight selected assets **equally** until the next rebalance.

This is a simple and transparent index methodology that is easy to explain, audit, and extend.

In [1]:
# If needed, uncomment and run:
# !pip install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

TypeError: expected string or bytes-like object, got 'NoneType'

## 1. Define the universe and backtest settings

We use ETFs as **asset-class proxies**:

- `SPY`: U.S. equities
- `TLT`: Long-term U.S. Treasuries
- `GLD`: Gold
- `VNQ`: U.S. REITs
- `DBC`: Broad commodities

You can easily replace or expand this universe later.

In [ ]:
# -----------------------------
# Backtest configuration
# -----------------------------
UNIVERSE = ["SPY", "TLT", "GLD", "VNQ", "DBC"]
BENCHMARK = "SPY"

START_DATE = "2012-01-01"
END_DATE = None            # None = up to latest available date

MOMENTUM_LOOKBACK = 126    # ~6 trading months
TOP_N = 3                  # number of assets selected each rebalance
REBALANCE_FREQ = "M"       # month-end rebalance
RISK_FREE_RATE = 0.0       # annualized, for Sharpe ratio

## 2. Download price data

We use **adjusted close prices** because they account for splits and dividends more appropriately for backtesting.

In [ ]:
# Download adjusted prices
raw = yf.download(
    tickers=UNIVERSE,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

# yfinance sometimes returns a multi-index column structure
if isinstance(raw.columns, pd.MultiIndex):
    if "Close" in raw.columns.get_level_values(0):
        prices = raw["Close"].copy()
    else:
        prices = raw.xs(raw.columns.levels[0][0], axis=1, level=0).copy()
else:
    prices = raw.copy()

# Keep only the requested tickers and drop rows with all NaNs
prices = prices[UNIVERSE].dropna(how="all")

# Forward-fill occasional missing values
prices = prices.ffill().dropna()

prices.tail()

## 3. Compute daily returns

The index will ultimately be built from daily asset returns, but weights will only change at monthly rebalance dates.

In [ ]:
daily_returns = prices.pct_change().fillna(0.0)
daily_returns.head()

## 4. Define the momentum signal

We use **trailing 6-month return**:

\[
\text{Momentum}_{t} = \frac{P_t}{P_{t-L}} - 1
\]

where:
- \( P_t \) is the latest price available at the rebalance date
- \( L \) is the lookback window (`126` trading days here)

To avoid look-ahead bias, the weights chosen on a rebalance date are based only on information available **up to that date**.

In [ ]:
momentum = prices.pct_change(MOMENTUM_LOOKBACK)
momentum.tail()

## 5. Create monthly rebalance dates

We rebalance at the **last trading day of each month**.

In [ ]:
# Last available trading day in each calendar month
rebalance_dates = prices.resample(REBALANCE_FREQ).last().index

# Keep only dates that exist in the trading calendar
rebalance_dates = [d for d in rebalance_dates if d in prices.index]

rebalance_dates[:5], rebalance_dates[-5:]

## 6. Build the index methodology

### Methodology
At each monthly rebalance:
1. Read the momentum score for each asset.
2. Rank assets from highest to lowest momentum.
3. Select the **top `TOP_N`** assets.
4. Assign **equal weights** to selected assets.
5. Hold those weights until the next rebalance.

If an asset has missing momentum on a rebalance date, it is excluded from selection for that date.

In [ ]:
def compute_rebalance_weights(momentum_df, rebalance_dates, top_n=3):
    '''
    Return a DataFrame of target weights on rebalance dates.
    '''
    weights = pd.DataFrame(
        0.0,
        index=pd.DatetimeIndex(rebalance_dates),
        columns=momentum_df.columns
    )

    for dt in weights.index:
        signal = momentum_df.loc[dt].dropna()
        if signal.empty:
            continue

        selected = signal.sort_values(ascending=False).head(top_n).index
        weights.loc[dt, selected] = 1.0 / len(selected)

    return weights

target_weights = compute_rebalance_weights(momentum, rebalance_dates, top_n=TOP_N)
target_weights.tail()

## 7. Expand target weights to daily holdings

The target weights are only defined on rebalance dates.  
To compute daily portfolio returns, we forward-fill weights between rebalance dates.

In [ ]:
# Reindex weights to all trading dates and forward-fill between rebalances
daily_weights = target_weights.reindex(prices.index).ffill().fillna(0.0)

# Shift by 1 day so that returns on day t are earned using weights decided at the close of day t-1
daily_weights = daily_weights.shift(1).fillna(0.0)

daily_weights.head()

## 8. Compute index returns

The custom index daily return is:

\[
r^{index}_t = \sum_i w_{i,t} \cdot r_{i,t}
\]

where:
- \( w_{i,t} \) is the weight in asset \( i \) on day \( t \)
- \( r_{i,t} \) is the asset's daily return

In [ ]:
strategy_returns = (daily_weights * daily_returns).sum(axis=1)
strategy_returns.name = "Custom Multi-Asset Index"

benchmark_returns = daily_returns[BENCHMARK].copy()
benchmark_returns.name = BENCHMARK

returns_df = pd.concat([strategy_returns, benchmark_returns], axis=1).dropna()
returns_df.head()

## 9. Convert returns into index levels

To make the series easier to compare visually, we convert each return stream into a normalized index level starting at **100**.

In [ ]:
def returns_to_index(returns, start_value=100.0):
    return start_value * (1 + returns).cumprod()

index_levels = pd.concat(
    [
        returns_to_index(returns_df["Custom Multi-Asset Index"]).rename("Custom Multi-Asset Index"),
        returns_to_index(returns_df[BENCHMARK]).rename(BENCHMARK)
    ],
    axis=1
)

index_levels.head()

## 10. Performance metrics

We compute:
- **Total Return**
- **CAGR**
- **Annualized Volatility**
- **Sharpe Ratio**
- **Maximum Drawdown**

### Sharpe Ratio
We use the standard approximation:

\[
\text{Sharpe} = \frac{\mu_p - r_f}{\sigma_p}
\]

where returns are annualized from daily data.

In [ ]:
def max_drawdown(index_series):
    running_max = index_series.cummax()
    drawdown = index_series / running_max - 1.0
    return drawdown.min()

def performance_summary(returns, rf=0.0, trading_days=252):
    returns = returns.dropna()
    index_series = returns_to_index(returns)

    total_return = index_series.iloc[-1] / index_series.iloc[0] - 1.0
    n_years = len(returns) / trading_days
    cagr = (index_series.iloc[-1] / index_series.iloc[0]) ** (1 / n_years) - 1.0 if n_years > 0 else np.nan
    ann_vol = returns.std() * np.sqrt(trading_days)
    ann_return = returns.mean() * trading_days
    sharpe = (ann_return - rf) / ann_vol if ann_vol != 0 else np.nan
    mdd = max_drawdown(index_series)

    return pd.Series({
        "Total Return": total_return,
        "CAGR": cagr,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": mdd
    })

summary = pd.concat(
    [
        performance_summary(returns_df["Custom Multi-Asset Index"], rf=RISK_FREE_RATE).rename("Custom Multi-Asset Index"),
        performance_summary(returns_df[BENCHMARK], rf=RISK_FREE_RATE).rename(BENCHMARK)
    ],
    axis=1
).T

summary

## 11. Rebalance history

This table shows which assets were held at each rebalance date and in what weights.

In [ ]:
rebalance_history = target_weights[target_weights.sum(axis=1) > 0].copy()
rebalance_history.tail(12)

## 12. Visualization: index performance vs SPY

In [ ]:
ax = index_levels.plot(title="Custom Momentum Index vs SPY", linewidth=2)
ax.set_ylabel("Index Level (Start = 100)")
plt.show()

## 13. Visualization: drawdowns

In [ ]:
def compute_drawdown_series(index_series):
    running_max = index_series.cummax()
    return index_series / running_max - 1.0

drawdowns = index_levels.apply(compute_drawdown_series)

ax = drawdowns.plot(title="Drawdowns", linewidth=2)
ax.set_ylabel("Drawdown")
plt.show()

## 14. Visualization: weights through time

In [ ]:
ax = daily_weights.plot.area(title="Daily Portfolio Weights", alpha=0.8)
ax.set_ylabel("Weight")
plt.show()

## 15. Visualization: rolling Sharpe ratio

This helps evaluate whether the strategy's risk-adjusted performance was stable over time.

In [ ]:
ROLLING_WINDOW = 252  # 1 year of trading days

rolling_sharpe = (
    (returns_df.rolling(ROLLING_WINDOW).mean() * 252 - RISK_FREE_RATE) /
    (returns_df.rolling(ROLLING_WINDOW).std() * np.sqrt(252))
)

ax = rolling_sharpe.plot(title="Rolling 1-Year Sharpe Ratio", linewidth=2)
ax.set_ylabel("Sharpe Ratio")
plt.show()

## 16. Interpretation

This project demonstrates a complete workflow for a **custom multi-asset index construction** problem:

- Use market data from `yfinance`
- Define a transparent **index methodology**
- Select assets using a **momentum factor**
- Rebalance **monthly**
- Measure performance against a benchmark (`SPY`)
- Evaluate both return and risk-adjusted metrics such as **Sharpe ratio**

## Possible extensions
You can improve or customize this framework by adding:
- Transaction costs
- Turnover analysis
- Volatility scaling
- Risk-parity weighting instead of equal weighting
- Absolute momentum filters (for example, move to cash if momentum is negative)
- More asset classes (international equity, short duration bonds, FX, crypto proxies, etc.)
- Walk-forward parameter testing

## 17. Optional: export results

In [ ]:
# Save outputs if desired
summary.to_csv("performance_summary.csv")
rebalance_history.to_csv("rebalance_history.csv")
index_levels.to_csv("index_levels.csv")

print("Saved: performance_summary.csv, rebalance_history.csv, index_levels.csv")

## 18. Final notes

This notebook is intentionally designed to be:
- Easy to explain in an interview or project presentation
- Easy to modify for more sophisticated index methodologies
- Clean enough to serve as a portfolio project

A good next step would be to package the main logic into reusable functions or a class so the strategy can be tested on multiple universes and parameter settings.